# MarshPulse · MarshNet V2 — Hawizeh Satellite Training

Real Sentinel-2 L2A satellite images → a residual ConvGRU-U-Net → chronological holdout. **No synthetic fallback.** This notebook will fail if internet imagery is unavailable or invalid. Target masks are MNDWI/NDVI-derived **proxies**, not ground-truth field labels or validated hydrological outcomes.

In [ ]:
%pip -q install "pystac-client>=0.8,<1" "planetary-computer>=1.0,<2" "rasterio>=1.3,<2" "numpy>=1.26,<3"

## 1 · Source code

The repository is public. Enable **Internet** in Kaggle notebook settings, or supply a clone as a Kaggle input dataset. A GitHub connection by itself does not grant this conversation permission to submit or start a Kaggle job.

In [ ]:
import sys, subprocess, json
from pathlib import Path
ROOT=Path("/kaggle/working/MarshPulse")
if not (ROOT/"ml/marshnet_v2.py").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/ahpa5246-lgtm/MarshPulse.git",str(ROOT)],check=True)
print("Source:",ROOT)

## 2 · Collect genuine Iraqi marsh imagery

The default **small rectangular pilot near Hawizeh Marsh** is not a surveyed marsh polygon. EPSG:4326 bounding rectangle: [47.55,31.50,47.65,31.60]. Sentinel acquisition records and cloud masks are preserved. Missing pixels remain invalid rather than becoming 'dry'.

In [ ]:
BBOX=[47.55,31.50,47.65,31.60]   # W,S,E,N — NOT the official Ramsar boundary
START,END="2023-01","2025-12"
SIZE=96   # Practical pilot raster dimensions; not 10m full-marsh analysis
OUTPUT=Path("/kaggle/working/marshpulse/sentinel")
subprocess.run([sys.executable,str(ROOT/"ml/collect_sentinel.py"),"--bbox",*map(str,BBOX),"--start",START,"--end",END,"--pixels",str(SIZE),"--max-scenes","2","--min-coverage","0.20","--out",str(OUTPUT)],check=True)

## 3 · Inspect provenance before training

Do not ignore missing seasons or cloud coverage. If the downloaded series is too incomplete, change the pilot area and explicitly document the change; never synthesize missing monthly water truth.

In [ ]:
import numpy as np
manifest=json.loads((OUTPUT/"satellite_manifest.json").read_text())
with np.load(OUTPUT/"hawizeh_s2_monthly.npz") as z:
    cube=z["x"]; dates=z["dates"]
print("Real-source manifest:",manifest["source"])
print("Pilot cube shape:",cube.shape,"time:",dates[0],dates[-1])
print("Acquisitions:",sum(len(v["scenes"]) for v in manifest["months"]))
print("Clear coverage:",[(a["month"],round(a["fraction_valid"],2)) for a in manifest["months"][:12]])
assert sum(len(v["scenes"]) for v in manifest["months"])>0

## 4 · Train MarshNet V2

Residual multiscale encoder → two ConvGRU layers → decoder with skip connections. Input is four spectral bands, NDVI, MNDWI and an observed-pixel flag, over 4 past months. It predicts next-month water-proxy probabilities. Evaluation is on later months, using **the same observed pixels for the persistence baseline**.

The checkpoint is a **PyTorch V2** model. It is not interchangeable with the website's browser-only M1 `marshnet.json` format.

In [ ]:
DEST=Path("/kaggle/working/marshpulse/training")
subprocess.run([sys.executable,str(ROOT/"ml/train_satellite.py"),"--input",str(OUTPUT/"hawizeh_s2_monthly.npz"),"--out",str(DEST),"--base","32","--lookback","4","--epochs","24","--batch","2"],check=True)

## 5 · Audit the results; do not assume success

Brier score is better if lower. A result worse than persistence is a negative result. Neither implies accuracy of a real hydraulic rescue plan.

In [ ]:
score=json.loads((DEST/"evaluation.json").read_text())
print("Data SHA-256:",score["dataset_sha256"])
print("Test period:",score["test_months"])
print("MarshNet V2:",score["model_same_pixels_as_baseline"])
print("Persistence:",score["persistence_same_pixels_as_model"])
if score["model_same_pixels_as_baseline"]["brier"]>=score["persistence_same_pixels_as_model"]["brier"]:
    print("NEGATIVE RESULT: did not beat persistence.")
else:
    print("Improvement on this pilot/threshold only; not field-ground-truth validation.")

## 6 · Export

Kaggle Output files: `sentinel/satellite_manifest.json`, `sentinel/hawizeh_s2_monthly.npz`, `training/marshnet_v2.pt`, `training/evaluation.json`, and `training/test_predictions.npz`. Keep all files and dataset SHA-256 together. A second region and independent ground-truth water labels are required before a scientific claim of generalization. Source: ESA Sentinel-2 via Microsoft Planetary Computer; Copernicus Sentinel Data Terms apply.